In [209]:
import numpy as np
import pandas as pd

In [210]:
path = "https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv"

In [211]:
df = pd.read_csv(path)
df =df.copy()
df.head()

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors,fuel_efficiency_mpg
0,170,3.0,159.0,3413.433759,17.7,2003,Europe,Gasoline,All-wheel drive,0.0,13.231729
1,130,5.0,97.0,3149.664934,17.8,2007,USA,Gasoline,Front-wheel drive,0.0,13.688217
2,170,NaN,78.0,3079.038997,15.1,2018,Europe,Gasoline,Front-wheel drive,0.0,14.246341
3,220,4.0,NaN,2542.392402,20.2,2009,USA,Diesel,All-wheel drive,2.0,16.912736
4,210,1.0,140.0,3460.870990,14.4,2009,Europe,Gasoline,All-wheel drive,2.0,12.488369


In [212]:
df = df[['engine_displacement','horsepower','vehicle_weight','model_year','fuel_efficiency_mpg']]
df.head()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,170,159.0,3413.433759,2003,13.231729
1,130,97.0,3149.664934,2007,13.688217
2,170,78.0,3079.038997,2018,14.246341
3,220,NaN,2542.392402,2009,16.912736
4,210,140.0,3460.870990,2009,12.488369


Question 1 :There's one column with missing values. What is it?

In [213]:
df.columns[df.isna().sum() > 0]

Index(['horsepower'], dtype='object')

Question 2: What's the median (50% percentile) for variable 'horsepower'?

In [214]:
df['horsepower'].describe()["50%"]

np.float64(149.0)

Prepare and split the dataset
 - Shuffle the dataset (the filtered one you created above), use seed 42.
 - Split your data in train/val/test sets, with 60%/20%/20% distribution

In [215]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

n, n_val, n_test , n_train

(9704, 1940, 1940, 5824)

In [216]:
idx = np.arange(n)
np.random.seed(42)
np.random.shuffle(idx)

In [217]:
df_train = df.iloc[idx[:n_train]]
df_test = df.iloc[idx[n_train: n_train + n_test]]
df_val = df.iloc[idx[n_train + n_test :]]

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

In [218]:
y_train = df_train['fuel_efficiency_mpg']
y_test = df_test['fuel_efficiency_mpg']
y_val = df_val['fuel_efficiency_mpg']

In [219]:
del df_train["fuel_efficiency_mpg"]
del df_test["fuel_efficiency_mpg"]
del df_val["fuel_efficiency_mpg"]

Question 3
- We need to deal with missing values for the column from Q1.
- We have two options: fill it with 0 or with the mean of this variable.
- Try both options. For each, train a linear regression model without regularization using the code from the - lessons.
- For computing the mean, use the training only!
- Use the validation dataset to evaluate the models and compare the RMSE of each option.
- Round the RMSE scores to 2 decimal digits using round(score, 2)

Which option gives better RMSE?

In [220]:
def train_linear_regression(X,y):
    m = len(X)
    ones = np.ones(m)
    X = np.column_stack([ones, X])
    
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    return w_full[0], w_full[1:]

In [221]:
# filling missing values with 0
def prepare_X(df):
    df = df.copy()
    X = df.fillna(0).values
    return X

In [222]:
X_train = prepare_X(df_train)
w0 , w = train_linear_regression(X_train, y_train)

X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)
y_pred

array([16.49729054, 14.97133662, 11.29273331, ..., 17.97612728,
       20.650853  , 16.97233137])

In [223]:
# rmse
def rmse(y, y_pred):
    se = (y - y_pred)**2
    mse = se.mean()
    return np.sqrt(mse)

In [224]:
score_zero = rmse(y_val, y_pred).round(2)
score_zero

np.float64(0.52)

In [225]:
# filling missing values with mean
def prepare_X(df):
    df = df.copy()
    mean = df['horsepower'].mean()
    X = df.fillna(mean).values
    return X

In [226]:
X_train = prepare_X(df_train)
w0 , w = train_linear_regression(X_train, y_train)

X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)
score_mean = rmse(y_val, y_pred).round(2)
score_mean

np.float64(0.46)

In [227]:
print(f"score_zero: {score_zero}")
print(f"score_mean: {score_mean}")

score_zero: 0.52
score_mean: 0.46


Question 4
- Now let's train a regularized linear regression.
- For this question, fill the NAs with 0.
- Try different values of r from this list: [0, 0.01, 0.1, 1, 5, 10, 100].
- Use RMSE to evaluate the model on the validation dataset.
- Round the RMSE scores to 2 decimal digits.
- Which r gives the best RMSE?

In [228]:
def train_linear_regression_reg(X,y, r):
    m = len(X)
    ones = np.ones(m)
    X = np.column_stack([ones, X])
    
    XTX = X.T.dot(X)
    XTX = XTX + r* np.eye(X.shape[1])
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    return w_full[0], w_full[1:]

In [229]:
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    X_train = prepare_X(df_train)
    w0 , w = train_linear_regression_reg(X_train, y_train, r=r)

    X_val = prepare_X(df_val)
    y_pred = w0 + X_val.dot(w)
    score = rmse(y_val, y_pred)
    print(r , w0, score)

0 28.9252599531686 0.4625016894306726
0.01 25.011482646475216 0.462565287595702
0.1 11.277820454364848 0.4656031407974988
1 1.7374758273521973 0.470254270571137
5 0.3650421910626702 0.4710910051885635
10 0.1836835983229268 0.47120468414440325
100 0.018480250159447004 0.4713088066156039


In [230]:
r = 0
X_train = prepare_X(df_train)
w0 , w = train_linear_regression_reg(X_train, y_train, r)

X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)
score_mean = rmse(y_val, y_pred)
score_mean

np.float64(0.4625016894306726)

Question 5
- We used seed 42 for splitting the data. Let's find out how selecting the seed influences our score.
- Try different seed values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9].
- For each seed, do the train/validation/test split with 60%/20%/20% distribution.
- Fill the missing values with 0 and train a model without regularization.
- For each seed, evaluate the model on the validation dataset and collect the RMSE scores.
- What's the standard deviation of all the scores? To compute the standard deviation, use np.std.
- Round the result to 3 decimal digits (round(std, 3))

What's the value of std?

In [231]:
scores = []
for i in range(10):
    idx = np.arange(n)
    np.random.seed(i)
    np.random.shuffle(idx)
    
    df_train = df.iloc[idx[:n_train]]
    df_test = df.iloc[idx[n_train: n_train + n_test]]
    df_val = df.iloc[idx[n_train + n_test :]]

    df_train = df_train.reset_index(drop=True)
    df_test = df_test.reset_index(drop=True)
    df_val = df_val.reset_index(drop=True)
    
    y_train = df_train.fuel_efficiency_mpg
    y_test = df_test.fuel_efficiency_mpg
    y_val = df_val.fuel_efficiency_mpg
    
    del df_train["fuel_efficiency_mpg"]
    del df_test['fuel_efficiency_mpg']
    del df_val["fuel_efficiency_mpg"]
    
    def prepare_X(df):
        df = df.fillna(0)
        X = df.values
        return X
    
    X_train = prepare_X(df_train)
    w0 , w = train_linear_regression(X_train, y_train)
    
    X_val = prepare_X(df_val)
    y_pred = w0 + X_val.dot(w)
    scores.append(rmse(y_val, y_pred))
    
    

In [232]:
print(scores)

[np.float64(0.5227776356502154), np.float64(0.528684011245838), np.float64(0.5107333458717226), np.float64(0.5193805167464705), np.float64(0.5324643389275652), np.float64(0.5084224078949868), np.float64(0.5258055823439493), np.float64(0.5118291291521008), np.float64(0.5077496159497334), np.float64(0.5158284328942937)]


In [233]:
np.std(scores).round(3)

np.float64(0.008)

Question 6
- Split the dataset like previously, use seed 9.
- Combine train and validation datasets.
- Fill the missing values with 0 and train a model with r=0.001.

What's the RMSE on the test dataset?

In [235]:
r=0.001
idx = np.arange(n)
np.random.seed(9)
np.random.shuffle(idx)
    
df_train = df.iloc[idx[:n_train]]
df_test = df.iloc[idx[n_train: n_train + n_test]]
df_val = df.iloc[idx[n_train + n_test :]]

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
    
y_train = df_train.fuel_efficiency_mpg
y_test = df_test.fuel_efficiency_mpg
y_val = df_val.fuel_efficiency_mpg
    
del df_train["fuel_efficiency_mpg"]
del df_test['fuel_efficiency_mpg']
del df_val["fuel_efficiency_mpg"]
    
def prepare_X(df):
    df = df.fillna(0)
    X = df.values
    return X
    
X_train = prepare_X(df_train)
w0 , w = train_linear_regression_reg(X_train, y_train, r)
    
X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)
rmse(y_val, y_pred)
    
    

np.float64(0.5158439492752485)